In [ ]:
import pandas as pd
import numpy as np

from vpop_calibration import *

%load_ext autoreload
%autoreload 2

In [ ]:
protocol_design = pd.DataFrame({"protocol_arm": ["dose-1", "dose-10"], "dose": [1, 10]})

struct_model = StructuralSbml(
    model_path="../../vpop_calibration/test/sbml/assets/model.xml",
    inputs=["k__12", "k__el", "dose", "k__a"],
    outputs=["A0", "A1", "A2"],
    protocol_design=protocol_design,
)

In [ ]:
ground_truth = {
    "pdu": {
        "k__12": {"prior": 0.1, "prior_omega": 0.2},
        "k__a": {"prior": 10.0, "prior_omega": 0.5},
        "k__el": {"prior": 1.0, "prior_omega": 0.5},
    },
    "error_model": {
        "A0": {"error_type": "additive", "sigma_add": 0.03},
        "A1": {"error_type": "additive", "sigma_add": 0.03},
        "A2": {"error_type": "additive", "sigma_add": 0.03},
    },
}
time: list[float] = list(np.linspace(0, 1e1, 20))
nb_patients = 100

obs_df = generate_synthetic_data(
    struct_model=struct_model,
    param_distrib=ground_truth,
    nb_patients=nb_patients,
    time=time,
)

In [ ]:
from plotnine import *

(
    ggplot(obs_df, aes(x="time", y="value", color="id"))
    + geom_line()
    + facet_grid(cols="output_name", rows="protocol_arm")
    + theme(legend_position="none")
)

In [ ]:
prior = {
    "model_intrinsic": {"k__12": {"prior": 0.5}},
    "pdu": {
        # "k__12": {"prior": 0.1, "prior_omega": 0.2},
        "k__a": {"prior": 10.0, "prior_omega": 0.5},
        "k__el": {"prior": 1.0, "prior_omega": 0.5},
    },
    "error_model": {
        "A0": {"error_type": "additive", "sigma_add": 0.01},
        "A1": {"error_type": "additive", "sigma_add": 0.01},
        "A2": {"error_type": "additive", "sigma_add": 0.01},
    },
}

config = Config(
    saem=SaemConfigDict(
        nb_iter_burnin=0,
        nb_iter_learning=100,
        nb_iter_smoothing=100,
        plot_frames=5,
    ),
    nlme=NlmeConfigDict(nb_chains=1),
)

nlme_model = NlmeModel(
    df=obs_df, prior_params=prior, structural_model=struct_model, config=config
)

In [ ]:
nlme_model.optimizer.run()

In [ ]:
nlme_model.diagnostics.sample_conditional_distribution(nb_samples=100)

In [ ]:
nlme_model.plot.map_estimates()

In [ ]:
nlme_model.plot.map_estimates_gof()

In [ ]:
nlme_model.plot.weighted_residuals("iwres")

In [ ]:
nlme_model.plot.vpc()

In [ ]:
nlme_model.plot.conditional_codistributions()